In [47]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
from implementations import (
    least_squares,
    ridge_regression,
    mean_squared_error_gd,
    mean_squared_error_sgd,
    logistic_regression,
    reg_logistic_regression,
    calculate_neg_ll_loss,
)
from utils import (
    load_preprocess,
    build_k_indices,
    threshold_curve,
    add_interactions,
    INTERACTION_FEATURES,
    f1_score,
    predict,
    step_size,
    weighted_reg_logistic_regression,
    add_powers
)
import warnings
warnings.filterwarnings("ignore", message=".*encountered in matmul")

In [48]:
QUICK = False
K= 5  
MAX_ITERS = 1000  
SGD_ITERS = 100_000  
LAMBDAS = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
POS_WEIGHTS = [1.5, 2, 3] # how much more a positive sample counts (weighted logistic)

os.makedirs("results", exist_ok=True)
np.random.seed(0) 

## Loading Data 

In [49]:
tx, _, y, _, features = load_preprocess()
if QUICK:
    keep = np.random.default_rng(0).choice(len(y), min(50_000, len(y)), replace=False)
    tx, y = tx[keep], y[keep]
y01 = (y == 1).astype(float)  # logistic regression needs labels 0/1
N, D = tx.shape
k_indices = build_k_indices(y, K, seed=1)
print(k_indices.shape)
w0 = np.zeros(D)
gamma_mse = step_size(tx, "mse")
gamma_sgd = 0.1 / np.mean(np.sum(tx**2, axis=1))  # one sample's curvature is ||x_n||^2
print(f"N={N}  D={D}  positives={y01.mean():.1%}  quick mode: {QUICK}")

(5, 65627)
N=328135  D=249  positives=8.8%  quick mode: False


## Cross-validation 

In [50]:
def cross_validation(y, y01, tx, k_indices, k, train):
    """Train on all folds except the k-th one (same split as lab 4).

    Args:
        y, y01:     labels in {-1, 1} and {0, 1}, shape=(N,)
        tx:         data matrix, shape=(N, D)
        k_indices:  2D array returned by build_k_indices()
        k:          scalar, the k-th fold is the validation fold
        train:      function train(y_pm, y01, tx) -> w

    Returns:
        w, te_indice, tr_indice: the weights and the validation / training row indices
    """
    te_indice = k_indices[k]
    tr_indice = k_indices[~(np.arange(k_indices.shape[0]) == k)].reshape(-1)
    w = train(y[tr_indice], y01[tr_indice], tx[tr_indice])
    return w, te_indice, tr_indice


def cross_validate(model, param, train):
    """Run cross_validation() for every fold, tune the threshold and summarize."""
    start = time.time()
    oof_scores = np.full(N, np.nan)  # each row scored by the model that did not see it
    fitted = []
    for k in range(K):
        try:
            w, te_indice, tr_indice = cross_validation(y, y01, tx, k_indices, k, train)
        except np.linalg.LinAlgError as err:  # e.g. least squares with a singular X^T X
            print(f"{model:<22} {param:<16} failed: {err}")
            return None
        oof_scores[te_indice] = tx[te_indice] @ w
        fitted.append((te_indice, tr_indice, w))

    used = k_indices.reshape(-1)  # rows in some fold (build_k_indices may leave out N % K rows)
    thresholds, f1s = threshold_curve(oof_scores[used], y[used])
    t = thresholds[np.argmax(f1s)]
    val_f1 = [f1_score(y[te], predict(oof_scores[te], t)) for te, _, _ in fitted]
    val_acc = [np.mean(predict(oof_scores[te], t) == y[te]) for te, _, _ in fitted]
    train_f1 = [f1_score(y[tr], predict(tx[tr] @ w, t)) for _, tr, w in fitted]

    r = dict(
        model=model,
        param=param,
        f1=np.mean(val_f1),
        f1_std=np.std(val_f1),
        acc=np.mean(val_acc),
        acc_std=np.std(val_acc),
        train_f1=np.mean(train_f1),
        threshold=t,
        curve=(thresholds, f1s),
    )
    print(
        f"{model:<22} {param:<16} F1 {r['f1']:.4f} ± {r['f1_std']:.4f}   acc {r['acc']:.4f}   "
        f"train F1 {r['train_f1']:.4f}   ({time.time() - start:.0f}s)"
    )
    return r

results = []

def run(model, param, train):
    """Cross-validate one setting and store its result."""
    r = cross_validate(model, param, train)
    if r is not None:
        results.append(r)
    return r

In [51]:
def random_undersample(x, y, und_ratio, seed):
    """ Randomly undersample the data to even out class imbalance:
        Randomly select only m samples from the MAJORITY class, where m equals the
        total number of samples in the dataset from the MINORITY class
    """
    labels, counts = np.unique(y, return_counts=True)
    m = counts.min()  # count of data points in minority class
    min_label = labels[counts.argmin()]  # minority class label
    maj_indices = np.where(y != min_label)[0]  # indices for data points in majority class
    min_indices = np.where(y == min_label)[0]  # indices for data points in minority class
    x_maj_only = x[maj_indices]  # dataset filtered for majority class only
    y_maj_only = y[maj_indices]  # dataset filtered for majority class only
    x_min_only = x[min_indices]  # dataset filtered for minority class only
    y_min_only = y[min_indices]  # dataset filtered for minority class only
    # create ndarray of randomized indices for undersampling
    # use argument for ratio of majority class labels to 1 minority class label
    und_rand_idx = np.random.permutation(m * und_ratio)
    # undersample majority class data points using randomly generated indices
    x_und = x_maj_only[und_rand_idx]
    y_und = y_maj_only[und_rand_idx]
    # print(x_und.shape)
    # print(y_und.shape)

    # randomly generate indices for final reshuffle
    recombine_rand_idx = np.random.permutation(und_rand_idx.shape[0] + m)
    # recombine with data points from the minority class
    x_recombine = np.concatenate([x_und, x_min_only])
    y_recombine = np.concatenate([y_und, y_min_only])
    
    return x_recombine[recombine_rand_idx], y_recombine[recombine_rand_idx]

In [52]:
# asawyer version - CLASS WEIGHT 3 WITH INTERACTIONS
# Trying 6 things with cross-validation to compare, including baseline:
# 0. baseline: no modifications (unregularized, underweighted logistic regression)
# 1. random undersampling at ratio 3:1, no threshold tuning 
# 2. class weight 3, no threshold tuning
# 3. baseline with threshold tuning (no weight modifications)
# 4. random undersampling at ratio 3:1 with threshold tuning
# 5. class weight 3 with threshold tuning
# For each, take average of F1 score, accuracy, and test loss across all folds. Then compare.

# class weight function for 2 and 5: weighted_reg_logistic_regression()


g = step_size(tx, "logistic")


def compute_accuracy(y_test, y_pred):
    """ Calculate accuracy: correct predictions over total predictions
        (TP + TN) / (TP + TN + FP + TN) """
    correct = np.sum(y_pred == y_test)
    total = y_pred.shape[0]
    
    return correct / total

    
def update_metrics(x_te, y_te, y01_te, w, thr):
    y_pred = predict(x_te @ w, thr)
    # 3d results array: indices are fold number (k), then approach index, then metric index
    f1 = f1_score(y_te, y_pred)  # F1
    acc = compute_accuracy(y_te, y_pred)  # accuracy
    te_loss = calculate_neg_ll_loss(y01_te, x_te, w)
    return [f1, acc, te_loss]


def get_f1_threshold(x_te, y_te, w):
    thresholds, f1s_thr = threshold_curve(x_te @ w, y_te)
    t = thresholds[np.argmax(f1s_thr)]
    return t

def cross_validation(y, y01, tx, w0, k_indices, k, res, und_seeds):
    """Train on all folds except the k-th one (same split as lab 4).

    Args:
        y, y01:     labels in {-1, 1} and {0, 1}, shape=(N,)
        tx:         data matrix, shape=(N, D)
        k_indices:  2D array returned by build_k_indices()
        k:          scalar, the k-th fold is the validation fold
        train:      function train(y_pm, y01, tx) -> w

    Returns:
        w, te_indice, tr_indice: the weights and the validation / training row indices
    """
    # get test and training subsets
    te_indice = k_indices[k]
    tr_indice = k_indices[~(np.arange(k_indices.shape[0]) == k)].reshape(-1)

    # try various different approaches
    # first, use F1 score baseline threshold of 0.5
    t_default = 0.5
    # other hyperparameters
    und_ratio = 3  # ratio for random undersampling
    pw = 3  # class weight value
    lamdba_ = 0.0001
    # store metrics F1, accuracy, and test loss to later compare
    
    # 0: baseline
    w_base, _ = logistic_regression(y01[tr_indice], tx[tr_indice], w0, MAX_ITERS, g)
    res[k, 0, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_base, t_default)
    
    # 1: random undersampling with ratio 3:1
    x_tr_und, y_tr_und = random_undersample(tx[tr_indice], y01[tr_indice], und_ratio, und_seeds[k])
    w_und, _ = logistic_regression(y_tr_und, x_tr_und, w0, MAX_ITERS, g)
    res[k, 1, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_und, t_default)
    
    # 2: class weight 3, lambda = 0
    w_cw = weighted_reg_logistic_regression(y01[tr_indice], tx[tr_indice], 0, w0, MAX_ITERS, g, pw)
    res[k, 2, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_cw, t_default)

    # 2.5: class weight 3, lambda = 0.0001
    w_reg_cw = weighted_reg_logistic_regression(y01[tr_indice], tx[tr_indice], lamdba_, w0, MAX_ITERS, g, pw)
    res[k, 3, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_reg_cw, t_default)

    # 3: baseline with threshold tuning 
    t_base_thr = get_f1_threshold(tx[te_indice], y[te_indice], w_base)
    res[k, 4, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_base, t_base_thr)

    # 4: random undersampling at ratio 3:1 with threshold tuning
    t_und_thr = get_f1_threshold(tx[te_indice], y[te_indice], w_und)
    res[k, 5, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_und, t_und_thr)
    
    # 5: class weight 3 with threshold tuning, , lambda = 0
    t_cw_thr = get_f1_threshold(tx[te_indice], y[te_indice], w_cw)
    res[k, 6, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_cw, t_cw_thr)

    # 5.5: class weight 3 with threshold tuning, , lambda = 0.0001
    t_reg_cw_thr = get_f1_threshold(tx[te_indice], y[te_indice], w_reg_cw)
    res[k, 7, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_reg_cw, t_reg_cw_thr)
    
    return res


def cross_validation_comparison_protocol(tx):
    """Run cross_validation() for every fold, tune the threshold and summarize."""
    # start = time.time()
    # oof_scores = np.full(N, np.nan)  # each row scored by the model that did not see it
    # fitted = []
    approaches = ["baseline", "undersample", "class_weight", "reg_class_weight", "thr_baseline", "thr_undersample", "thr_class_weight", "thr_reg_class_weight"]
    metrics = ["f1", "accuracy", "te_loss"]
    results_interactions_cw3 = np.zeros((K, len(approaches), len(metrics)))
    und_seeds = [1,2,3,4,5]  # seeds for random undersampling

    # add interaction features
    tx_interactions, _, _ = add_interactions(tx, features)
    print(f"{tx_interactions.shape[1] - tx.shape[1]} interaction features added, {tx_interactions.shape[1]} in total")
    D = tx_interactions.shape[1]
    w0 = np.zeros(D)
    for k in range(K):
        results_interactions_cw3 = cross_validation(y, y01, tx_interactions, w0, k_indices, k, results_interactions_cw3, und_seeds)

    # get means and stds over all K folds, and compare
    mean = results_interactions_cw3.mean(axis=0)
    std = results_interactions_cw3.std(axis=0, ddof=1)

    for i, appr in enumerate(approaches):
        print(f"{appr}:\tF1: {mean[i, 0]:.5f} ± {std[i, 0]:.5f}\tAccuracy: {mean[i, 1]:.5f} ± {std[i, 1]:.5f}\tTest loss: {mean[i, 2]:.5f} ± {std[i, 2]:.5f}")

cross_validation_comparison_protocol(tx)

136 interaction features added, 385 in total
baseline:	F1: 0.09055 ± 0.00418	Accuracy: 0.91372 ± 0.00124	Test loss: 0.22105 ± 0.00251
undersample:	F1: 0.33076 ± 0.00844	Accuracy: 0.90777 ± 0.00156	Test loss: 0.27402 ± 0.00149
class_weight:	F1: 0.29876 ± 0.00737	Accuracy: 0.91027 ± 0.00132	Test loss: 0.26132 ± 0.00112
reg_class_weight:	F1: 0.29822 ± 0.00738	Accuracy: 0.91028 ± 0.00130	Test loss: 0.26142 ± 0.00111
thr_baseline:	F1: 0.43043 ± 0.00221	Accuracy: 0.86939 ± 0.00552	Test loss: 0.22105 ± 0.00251
thr_undersample:	F1: 0.42220 ± 0.00226	Accuracy: 0.86809 ± 0.00446	Test loss: 0.27402 ± 0.00149
thr_class_weight:	F1: 0.42492 ± 0.00162	Accuracy: 0.86928 ± 0.00348	Test loss: 0.26132 ± 0.00112
thr_reg_class_weight:	F1: 0.42492 ± 0.00166	Accuracy: 0.87101 ± 0.00318	Test loss: 0.26142 ± 0.00111


In [53]:
# asawyer version - CLASS WEIGHT 2 WITH INTERACTIONS
# Trying 6 things with cross-validation to compare, including baseline:
# 0. baseline: no modifications (unregularized, underweighted logistic regression)
# 1. random undersampling at ratio 3:1, no threshold tuning 
# 2. class weight 3, no threshold tuning
# 3. baseline with threshold tuning (no weight modifications)
# 4. random undersampling at ratio 3:1 with threshold tuning
# 5. class weight 3 with threshold tuning
# For each, take average of F1 score, accuracy, and test loss across all folds. Then compare.

# class weight function for 2 and 5: weighted_reg_logistic_regression()


g = step_size(tx, "logistic")


def compute_accuracy(y_test, y_pred):
    """ Calculate accuracy: correct predictions over total predictions
        (TP + TN) / (TP + TN + FP + TN) """
    correct = np.sum(y_pred == y_test)
    total = y_pred.shape[0]
    
    return correct / total

    
def update_metrics(x_te, y_te, y01_te, w, thr):
    y_pred = predict(x_te @ w, thr)
    # 3d results array: indices are fold number (k), then approach index, then metric index
    f1 = f1_score(y_te, y_pred)  # F1
    acc = compute_accuracy(y_te, y_pred)  # accuracy
    te_loss = calculate_neg_ll_loss(y01_te, x_te, w)
    return [f1, acc, te_loss]


def get_f1_threshold(x_te, y_te, w):
    thresholds, f1s_thr = threshold_curve(x_te @ w, y_te)
    t = thresholds[np.argmax(f1s_thr)]
    return t

def cross_validation(y, y01, tx, w0, k_indices, k, res, und_seeds):
    """Train on all folds except the k-th one (same split as lab 4).

    Args:
        y, y01:     labels in {-1, 1} and {0, 1}, shape=(N,)
        tx:         data matrix, shape=(N, D)
        k_indices:  2D array returned by build_k_indices()
        k:          scalar, the k-th fold is the validation fold
        train:      function train(y_pm, y01, tx) -> w

    Returns:
        w, te_indice, tr_indice: the weights and the validation / training row indices
    """
    # get test and training subsets
    te_indice = k_indices[k]
    tr_indice = k_indices[~(np.arange(k_indices.shape[0]) == k)].reshape(-1)

    # try various different approaches
    # first, use F1 score baseline threshold of 0.5
    t_default = 0.5
    # other hyperparameters
    und_ratio = 3  # ratio for random undersampling
    pw = 2  # class weight value
    lamdba_ = 0.0001
    # store metrics F1, accuracy, and test loss to later compare
    
    # 0: baseline
    w_base, _ = logistic_regression(y01[tr_indice], tx[tr_indice], w0, MAX_ITERS, g)
    res[k, 0, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_base, t_default)
    
    # 1: random undersampling with ratio 3:1
    x_tr_und, y_tr_und = random_undersample(tx[tr_indice], y01[tr_indice], und_ratio, und_seeds[k])
    w_und, _ = logistic_regression(y_tr_und, x_tr_und, w0, MAX_ITERS, g)
    res[k, 1, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_und, t_default)
    
    # 2: class weight 3, lambda = 0
    w_cw = weighted_reg_logistic_regression(y01[tr_indice], tx[tr_indice], 0, w0, MAX_ITERS, g, pw)
    res[k, 2, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_cw, t_default)

    # 2.5: class weight 3, lambda = 0.0001
    w_reg_cw = weighted_reg_logistic_regression(y01[tr_indice], tx[tr_indice], lamdba_, w0, MAX_ITERS, g, pw)
    res[k, 3, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_reg_cw, t_default)

    # 3: baseline with threshold tuning 
    t_base_thr = get_f1_threshold(tx[te_indice], y[te_indice], w_base)
    res[k, 4, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_base, t_base_thr)

    # 4: random undersampling at ratio 3:1 with threshold tuning
    t_und_thr = get_f1_threshold(tx[te_indice], y[te_indice], w_und)
    res[k, 5, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_und, t_und_thr)
    
    # 5: class weight 3 with threshold tuning, , lambda = 0
    t_cw_thr = get_f1_threshold(tx[te_indice], y[te_indice], w_cw)
    res[k, 6, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_cw, t_cw_thr)

    # 5.5: class weight 3 with threshold tuning, , lambda = 0.0001
    t_reg_cw_thr = get_f1_threshold(tx[te_indice], y[te_indice], w_reg_cw)
    res[k, 7, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_reg_cw, t_reg_cw_thr)
    
    return res


def cross_validation_comparison_protocol(tx):
    """Run cross_validation() for every fold, tune the threshold and summarize."""
    # start = time.time()
    # oof_scores = np.full(N, np.nan)  # each row scored by the model that did not see it
    # fitted = []
    approaches = ["baseline", "undersample", "class_weight", "reg_class_weight", "thr_baseline", "thr_undersample", "thr_class_weight", "thr_reg_class_weight"]
    metrics = ["f1", "accuracy", "te_loss"]
    results_interactions_cw2 = np.zeros((K, len(approaches), len(metrics)))
    und_seeds = [1,2,3,4,5]  # seeds for random undersampling

    # add interaction features
    tx_interactions, _, _ = add_interactions(tx, features)
    print(f"{tx_interactions.shape[1] - tx.shape[1]} interaction features added, {tx_interactions.shape[1]} in total")
    D = tx_interactions.shape[1]
    w0 = np.zeros(D)
    for k in range(K):
        results_interactions_cw2 = cross_validation(y, y01, tx_interactions, w0, k_indices, k, results_interactions_cw2, und_seeds)

    # get means and stds over all K folds, and compare
    mean = results_interactions_cw2.mean(axis=0)
    std = results_interactions_cw2.std(axis=0, ddof=1)

    for i, appr in enumerate(approaches):
        print(f"{appr}:\tF1: {mean[i, 0]:.5f} ± {std[i, 0]:.5f}\tAccuracy: {mean[i, 1]:.5f} ± {std[i, 1]:.5f}\tTest loss: {mean[i, 2]:.5f} ± {std[i, 2]:.5f}")

cross_validation_comparison_protocol(tx)

136 interaction features added, 385 in total
baseline:	F1: 0.09055 ± 0.00418	Accuracy: 0.91372 ± 0.00124	Test loss: 0.22105 ± 0.00251
undersample:	F1: 0.33076 ± 0.00844	Accuracy: 0.90777 ± 0.00156	Test loss: 0.27402 ± 0.00149
class_weight:	F1: 0.27118 ± 0.00516	Accuracy: 0.91355 ± 0.00154	Test loss: 0.23849 ± 0.00168
reg_class_weight:	F1: 0.27114 ± 0.00525	Accuracy: 0.91355 ± 0.00156	Test loss: 0.23858 ± 0.00168
thr_baseline:	F1: 0.43043 ± 0.00221	Accuracy: 0.86939 ± 0.00552	Test loss: 0.22105 ± 0.00251
thr_undersample:	F1: 0.42220 ± 0.00226	Accuracy: 0.86809 ± 0.00446	Test loss: 0.27402 ± 0.00149
thr_class_weight:	F1: 0.43033 ± 0.00281	Accuracy: 0.86879 ± 0.00744	Test loss: 0.23849 ± 0.00168
thr_reg_class_weight:	F1: 0.43070 ± 0.00277	Accuracy: 0.86887 ± 0.00746	Test loss: 0.23858 ± 0.00168


In [54]:
# asawyer version - CLASS WEIGHT 3 - performs a hair better than 2, at least w/o interactions
# Trying 6 things with cross-validation to compare, including baseline:
# 0. baseline: no modifications (unregularized, underweighted logistic regression)
# 1. random undersampling at ratio 3:1, no threshold tuning 
# 2. class weight 3, no threshold tuning
# 3. baseline with threshold tuning (no weight modifications)
# 4. random undersampling at ratio 3:1 with threshold tuning
# 5. class weight 3 with threshold tuning
# For each, take average of F1 score, accuracy, and test loss across all folds. Then compare.

# class weight function for 2 and 5: weighted_reg_logistic_regression()


g = step_size(tx, "logistic")


def compute_accuracy(y_test, y_pred):
    """ Calculate accuracy: correct predictions over total predictions
        (TP + TN) / (TP + TN + FP + TN) """
    correct = np.sum(y_pred == y_test)
    total = y_pred.shape[0]
    
    return correct / total

    
def update_metrics(x_te, y_te, y01_te, w, thr):
    y_pred = predict(x_te @ w, thr)
    # 3d results array: indices are fold number (k), then approach index, then metric index
    f1 = f1_score(y_te, y_pred)  # F1
    acc = compute_accuracy(y_te, y_pred)  # accuracy
    te_loss = calculate_neg_ll_loss(y01_te, x_te, w)
    return [f1, acc, te_loss]


def get_f1_threshold(x_te, y_te, w):
    thresholds, f1s_thr = threshold_curve(x_te @ w, y_te)
    t = thresholds[np.argmax(f1s_thr)]
    return t

def cross_validation(y, y01, tx, k_indices, k, res, und_seeds):
    """Train on all folds except the k-th one (same split as lab 4).

    Args:
        y, y01:     labels in {-1, 1} and {0, 1}, shape=(N,)
        tx:         data matrix, shape=(N, D)
        k_indices:  2D array returned by build_k_indices()
        k:          scalar, the k-th fold is the validation fold
        train:      function train(y_pm, y01, tx) -> w

    Returns:
        w, te_indice, tr_indice: the weights and the validation / training row indices
    """
    # get test and training subsets
    te_indice = k_indices[k]
    tr_indice = k_indices[~(np.arange(k_indices.shape[0]) == k)].reshape(-1)

    # try various different approaches
    # first, use F1 score baseline threshold of 0.5
    t_default = 0.5
    # other hyperparameters
    und_ratio = 3  # ratio for random undersampling
    pw = 3  # class weight value
    lamdba_ = 0.0001
    # store metrics F1, accuracy, and test loss to later compare
    
    # 0: baseline
    w_base, _ = logistic_regression(y01[tr_indice], tx[tr_indice], w0, MAX_ITERS, g)
    res[k, 0, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_base, t_default)
    
    # 1: random undersampling with ratio 3:1
    x_tr_und, y_tr_und = random_undersample(tx[tr_indice], y01[tr_indice], und_ratio, und_seeds[k])
    w_und, _ = logistic_regression(y_tr_und, x_tr_und, w0, MAX_ITERS, g)
    res[k, 1, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_und, t_default)
    
    # 2: class weight 3, lambda = 0
    w_cw = weighted_reg_logistic_regression(y01[tr_indice], tx[tr_indice], 0, w0, MAX_ITERS, g, pw)
    res[k, 2, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_cw, t_default)

    # 2.5: class weight 3, lambda = 0.0001
    w_reg_cw = weighted_reg_logistic_regression(y01[tr_indice], tx[tr_indice], lamdba_, w0, MAX_ITERS, g, pw)
    res[k, 3, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_reg_cw, t_default)

    # 3: baseline with threshold tuning 
    t_base_thr = get_f1_threshold(tx[te_indice], y[te_indice], w_base)
    res[k, 4, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_base, t_base_thr)

    # 4: random undersampling at ratio 3:1 with threshold tuning
    t_und_thr = get_f1_threshold(tx[te_indice], y[te_indice], w_und)
    res[k, 5, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_und, t_und_thr)
    
    # 5: class weight 3 with threshold tuning, , lambda = 0
    t_cw_thr = get_f1_threshold(tx[te_indice], y[te_indice], w_cw)
    res[k, 6, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_cw, t_cw_thr)

    # 5.5: class weight 3 with threshold tuning, , lambda = 0.0001
    t_reg_cw_thr = get_f1_threshold(tx[te_indice], y[te_indice], w_reg_cw)
    res[k, 7, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_reg_cw, t_reg_cw_thr)
    
    return res


def cross_validation_comparison_protocol():
    """Run cross_validation() for every fold, tune the threshold and summarize."""
    approaches = ["baseline", "undersample", "class_weight", "reg_class_weight", "thr_baseline", "thr_undersample", "thr_class_weight", "thr_reg_class_weight"]
    metrics = ["f1", "accuracy", "te_loss"]
    results_cw3 = np.zeros((K, len(approaches), len(metrics)))
    und_seeds = [1,2,3,4,5]  # seeds for random undersampling
    for k in range(K):
        results_cw3 = cross_validation(y, y01, tx, k_indices, k, results_cw3, und_seeds)

    # get means and stds over all K folds, and compare
    mean = results_cw3.mean(axis=0)
    std = results_cw3.std(axis=0, ddof=1)

    for i, appr in enumerate(approaches):
        print(f"{appr}:\tF1: {mean[i, 0]:.5f} ± {std[i, 0]:.5f}\tAccuracy: {mean[i, 1]:.5f} ± {std[i, 1]:.5f}\tTest loss: {mean[i, 2]:.5f} ± {std[i, 2]:.5f}")

cross_validation_comparison_protocol()

baseline:	F1: 0.12408 ± 0.00344	Accuracy: 0.91427 ± 0.00121	Test loss: 0.22237 ± 0.00290
undersample:	F1: 0.38395 ± 0.00504	Accuracy: 0.90367 ± 0.00148	Test loss: 0.28392 ± 0.00193
class_weight:	F1: 0.36447 ± 0.00452	Accuracy: 0.90762 ± 0.00161	Test loss: 0.26989 ± 0.00167
reg_class_weight:	F1: 0.36433 ± 0.00433	Accuracy: 0.90764 ± 0.00158	Test loss: 0.26998 ± 0.00167
thr_baseline:	F1: 0.42667 ± 0.00343	Accuracy: 0.87370 ± 0.00381	Test loss: 0.22237 ± 0.00290
thr_undersample:	F1: 0.42669 ± 0.00301	Accuracy: 0.87082 ± 0.00353	Test loss: 0.28392 ± 0.00193
thr_class_weight:	F1: 0.42707 ± 0.00361	Accuracy: 0.87148 ± 0.00551	Test loss: 0.26989 ± 0.00167
thr_reg_class_weight:	F1: 0.42706 ± 0.00376	Accuracy: 0.87090 ± 0.00678	Test loss: 0.26998 ± 0.00167


In [55]:
# asawyer version - CLASS WEIGHT 2
# Trying 6 things with cross-validation to compare, including baseline:
# 0. baseline: no modifications (unregularized, underweighted logistic regression)
# 1. random undersampling at ratio 3:1, no threshold tuning 
# 2. class weight 3, no threshold tuning
# 3. baseline with threshold tuning (no weight modifications)
# 4. random undersampling at ratio 3:1 with threshold tuning
# 5. class weight 3 with threshold tuning
# For each, take average of F1 score, accuracy, and test loss across all folds. Then compare.

# class weight function for 3 and 6: weighted_reg_logistic_regression()


g = step_size(tx, "logistic")


def compute_accuracy(y_test, y_pred):
    """ Calculate accuracy: correct predictions over total predictions
        (TP + TN) / (TP + TN + FP + TN) """
    correct = np.sum(y_pred == y_test)
    total = y_pred.shape[0]
    
    return correct / total

    
def update_metrics(x_te, y_te, y01_te, w, thr):
    y_pred = predict(x_te @ w, thr)
    # 3d results array: indices are fold number (k), then approach index, then metric index
    f1 = f1_score(y_te, y_pred)  # F1
    acc = compute_accuracy(y_te, y_pred)  # accuracy
    te_loss = calculate_neg_ll_loss(y01_te, x_te, w)
    return [f1, acc, te_loss]


def get_f1_threshold(x_te, y_te, w):
    thresholds, f1s_thr = threshold_curve(x_te @ w, y_te)
    t = thresholds[np.argmax(f1s_thr)]
    return t

def cross_validation(y, y01, tx, k_indices, k, res, und_seeds):
    """Train on all folds except the k-th one (same split as lab 4).

    Args:
        y, y01:     labels in {-1, 1} and {0, 1}, shape=(N,)
        tx:         data matrix, shape=(N, D)
        k_indices:  2D array returned by build_k_indices()
        k:          scalar, the k-th fold is the validation fold
        train:      function train(y_pm, y01, tx) -> w

    Returns:
        w, te_indice, tr_indice: the weights and the validation / training row indices
    """
    # get test and training subsets
    te_indice = k_indices[k]
    tr_indice = k_indices[~(np.arange(k_indices.shape[0]) == k)].reshape(-1)

    # try various different approaches
    # first, use F1 score baseline threshold of 0.5
    t_default = 0.5
    # other hyperparameters
    und_ratio = 3  # ratio for random undersampling
    pw = 2  # class weight value
    lamdba_ = 0.0001
    # store metrics F1, accuracy, and test loss to later compare
    
    # 0: baseline
    w_base, _ = logistic_regression(y01[tr_indice], tx[tr_indice], w0, MAX_ITERS, g)
    res[k, 0, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_base, t_default)
    
    # 1: random undersampling with ratio 3:1
    x_tr_und, y_tr_und = random_undersample(tx[tr_indice], y01[tr_indice], und_ratio, und_seeds[k])
    w_und, _ = logistic_regression(y_tr_und, x_tr_und, w0, MAX_ITERS, g)
    res[k, 1, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_und, t_default)
    
    # 2: class weight 3, lambda = 0
    w_cw = weighted_reg_logistic_regression(y01[tr_indice], tx[tr_indice], 0, w0, MAX_ITERS, g, pw)
    res[k, 2, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_cw, t_default)

    # 2.5: class weight 3, lambda = 0.0001
    w_reg_cw = weighted_reg_logistic_regression(y01[tr_indice], tx[tr_indice], lamdba_, w0, MAX_ITERS, g, pw)
    res[k, 3, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_reg_cw, t_default)

    # 3: baseline with threshold tuning 
    t_base_thr = get_f1_threshold(tx[te_indice], y[te_indice], w_base)
    res[k, 4, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_base, t_base_thr)

    # 4: random undersampling at ratio 3:1 with threshold tuning
    t_und_thr = get_f1_threshold(tx[te_indice], y[te_indice], w_und)
    res[k, 5, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_und, t_und_thr)
    
    # 5: class weight 3 with threshold tuning, , lambda = 0
    t_cw_thr = get_f1_threshold(tx[te_indice], y[te_indice], w_cw)
    res[k, 6, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_cw, t_cw_thr)

    # 5.5: class weight 3 with threshold tuning, , lambda = 0.0001
    t_reg_cw_thr = get_f1_threshold(tx[te_indice], y[te_indice], w_reg_cw)
    res[k, 7, :] = update_metrics(tx[te_indice], y[te_indice], y01[te_indice], w_reg_cw, t_reg_cw_thr)
    
    return res


def cross_validation_comparison_protocol():
    """Run cross_validation() for every fold, tune the threshold and summarize."""
    approaches = ["baseline", "undersample", "class_weight", "reg_class_weight", "thr_baseline", "thr_undersample", "thr_class_weight", "thr_reg_class_weight"]
    metrics = ["f1", "accuracy", "te_loss"]
    results_cw2 = np.zeros((K, len(approaches), len(metrics)))
    und_seeds = [1,2,3,4,5]  # seeds for random undersampling
    for k in range(K):
        results_cw2 = cross_validation(y, y01, tx, k_indices, k, results_cw2, und_seeds)

    # get means and stds over all K folds, and compare
    mean = results_cw2.mean(axis=0)
    std = results_cw2.std(axis=0, ddof=1)

    for i, appr in enumerate(approaches):
        print(f"{appr}:\tF1: {mean[i, 0]:.5f} ± {std[i, 0]:.5f}\tAccuracy: {mean[i, 1]:.5f} ± {std[i, 1]:.5f}\tTest loss: {mean[i, 2]:.5f} ± {std[i, 2]:.5f}")

cross_validation_comparison_protocol()

baseline:	F1: 0.12408 ± 0.00344	Accuracy: 0.91427 ± 0.00121	Test loss: 0.22237 ± 0.00290
undersample:	F1: 0.38395 ± 0.00504	Accuracy: 0.90367 ± 0.00148	Test loss: 0.28392 ± 0.00193
class_weight:	F1: 0.28378 ± 0.00356	Accuracy: 0.91361 ± 0.00171	Test loss: 0.24003 ± 0.00209
reg_class_weight:	F1: 0.28342 ± 0.00336	Accuracy: 0.91363 ± 0.00167	Test loss: 0.24011 ± 0.00209
thr_baseline:	F1: 0.42667 ± 0.00343	Accuracy: 0.87370 ± 0.00381	Test loss: 0.22237 ± 0.00290
thr_undersample:	F1: 0.42669 ± 0.00301	Accuracy: 0.87082 ± 0.00353	Test loss: 0.28392 ± 0.00193
thr_class_weight:	F1: 0.42692 ± 0.00348	Accuracy: 0.87090 ± 0.00875	Test loss: 0.24003 ± 0.00209
thr_reg_class_weight:	F1: 0.42685 ± 0.00365	Accuracy: 0.87261 ± 0.00749	Test loss: 0.24011 ± 0.00209


## Models

In [57]:
print(g)

0.3592936348456576


In [56]:
# Least squares and ridge regression for every lambda
run("least squares", "-", lambda yp, y01_, x: least_squares(yp, x)[0])
for lam in LAMBDAS:
    run("ridge", f"lambda={lam:g}", lambda yp, y01_, x, lam=lam: ridge_regression(yp, x, lam)[0])

TypeError: cross_validation() missing 1 required positional argument: 'und_seeds'

In [ ]:
# Linear regression with gradient descent and stochastic gradient descent
run(
    "MSE GD",
    f"gamma={gamma_mse:.3g}",
    lambda yp, y01_, x: mean_squared_error_gd(yp, x, w0, MAX_ITERS, gamma_mse)[0],
)
run(
    "MSE SGD",
    f"gamma={gamma_sgd:.3g}",
    lambda yp, y01_, x: mean_squared_error_sgd(yp, x, w0, SGD_ITERS, gamma_sgd)[0],
)

In [ ]:
# Logistic regression and regularized logistic regression for every lambda
g = step_size(tx, "logistic")
run("logistic", "-", lambda yp, y01_, x, g=g: logistic_regression(y01_, x, w0, MAX_ITERS, g)[0])

for lam in LAMBDAS:
    g = step_size(tx, "logistic", lam)
    run(
        "reg logistic",
        f"lambda={lam:g}",
        lambda yp, y01_, x, lam=lam, g=g: reg_logistic_regression(y01_, x, lam, w0, MAX_ITERS, g)[0],
    )

In [ ]:
# Weighted logistic regression for every lambda and positive weight
# uses best lambda from regularized logistic regression
best_reg = max((r for r in results if r["model"] == "reg logistic"), key=lambda r: r["f1"])
best_lam = float(best_reg["param"].split("=")[1])

for pw in POS_WEIGHTS:
    g = step_size(tx, "logistic", best_lam, y01, pw)
    run(
        "weighted reg logistic",
        f"lambda={best_lam:g} pw={pw}",
        lambda yp, y01_, x, pw=pw, g=g: weighted_reg_logistic_regression(
            y01_, x, best_lam, w0, MAX_ITERS, g, pw
        ),
    )

### Polynomial Features

In [ ]:
# Polynomial features: x^2, ..., x^degree for the non-binary columns (like build_poly in lab 4),
# on the best regularized logistic setting found above.
DEGREES = [2, 3]

# keep the class weight only if it beat plain regularized logistic regression
best_w = max((r for r in results if r["model"] == "weighted reg logistic"), key=lambda r: r["f1"])
best_pw = float(best_w["param"].split("pw=")[1]) if best_w["f1"] > best_reg["f1"] else 1.0

tx_base = tx
try:
    for degree in DEGREES:
        tx, _, _ = add_powers(tx_base, features, degree)  # cross_validate() uses the global tx
        print(f"degree {degree}: {tx.shape[1]} features")
        g = step_size(tx, "logistic", best_lam, y01, best_pw)
        w0_poly = np.zeros(tx.shape[1])
        run(
            "poly reg logistic",
            f"lambda={best_lam:g} pw={best_pw:g} degree={degree}",
            lambda yp, y01_, x, g=g, w0_poly=w0_poly: weighted_reg_logistic_regression(
                y01_, x, best_lam, w0_poly, MAX_ITERS, g, best_pw
            ),
        )
finally:
    tx = tx_base  # back to the original features for the next cells

### Interaction features 

In [ ]:
# Interaction features: products x_i * x_j of the main risk factors on the best regularized logistic setting found above.
best_lam, best_pw = 1e-4, 2.0  

used = [n for n in INTERACTION_FEATURES if n in features]
print(f"{len(used)} features combined pairwise: {used}")
print(f"not found after preprocessing: {[n for n in INTERACTION_FEATURES if n not in features]}")

tx_base = tx
try:
    tx, _, _ = add_interactions(tx_base, features)  # cross_validate() uses the global tx
    print(f"{tx.shape[1] - tx_base.shape[1]} interaction features added, {tx.shape[1]} in total")
    g = step_size(tx, "logistic", best_lam, y01, best_pw)
    w0_inter = np.zeros(tx.shape[1])
    run(
        "interact reg logistic",
        f"lambda={best_lam:g} pw={best_pw:g} interactions=1",
        lambda yp, y01_, x, g=g, w0_inter=w0_inter: weighted_reg_logistic_regression(
            y01_, x, best_lam, w0_inter, MAX_ITERS, g, best_pw
        ),
    )
finally:
    tx = tx_base  

In [ ]:
print(f"THRESHOLD = {results[-1]['threshold']:.6f}")

## Results

In [ ]:
with open("results/cv_results.csv", "w") as f:
    f.write("model,param,val_f1,val_f1_std,val_acc,val_acc_std,train_f1,threshold\n")
    for r in results:
        f.write(
            f"{r['model']},{r['param']},{r['f1']:.4f},{r['f1_std']:.4f},{r['acc']:.4f},"
            f"{r['acc_std']:.4f},{r['train_f1']:.4f},{r['threshold']:.6f}\n"
        )

print(f"{'model':<22} {'param':<16} {'val F1':>16} {'val acc':>8} {'train F1':>9}")
for r in sorted(results, key=lambda r: -r["f1"]):
    print(
        f"{r['model']:<22} {r['param']:<16} {r['f1']:.4f} ± {r['f1_std']:.4f} "
        f"{r['acc']:>8.4f} {r['train_f1']:>9.4f}"
    )

## Figure: model comparison

Best hyperparameters of each model, with the tuned threshold.

In [ ]:
# best version of each model, sorted by validation F1
families = list(dict.fromkeys(r["model"] for r in results))
bests = [max((r for r in results if r["model"] == m), key=lambda r: r["f1"]) for m in families]
bests.sort(key=lambda r: r["f1"])

fig, ax = plt.subplots(figsize=(8, 4.5))
y_pos = np.arange(len(bests))
colors = ["tab:orange" if "logistic" in r["model"] else "tab:blue" for r in bests]
ax.errorbar(
    [r["f1"] for r in bests],
    y_pos,
    xerr=[r["f1_std"] for r in bests],
    fmt="none",
    ecolor="gray",
    capsize=4,
)
ax.scatter([r["f1"] for r in bests], y_pos, c=colors, s=60, zorder=3)
for i, r in enumerate(bests):
    ax.text(r["f1"] + r["f1_std"] + 0.002, i, f"{r['f1']:.3f}", va="center")
ax.set_yticks(y_pos, [r["model"] for r in bests])
lo = min(r["f1"] - r["f1_std"] for r in bests)
hi = max(r["f1"] + r["f1_std"] for r in bests)
ax.set_xlim(lo - 0.005, hi + 0.012)  # zoom on the range of the results
ax.set_xlabel(f"validation F1 ({K}-fold CV, mean ± std)")
ax.set_title("Model comparison (best hyperparameters, tuned threshold)")
ax.scatter([], [], c="tab:blue", label="MSE loss")
ax.scatter([], [], c="tab:orange", label="logistic loss")
ax.legend(loc="upper left")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig("results/model_comparison.png", dpi=150)
plt.show()

## Figure: F1 vs $\lambda$ (bias–variance)

- **Train much higher than validation**: overfitting (small $\lambda$).
- **Both dropping**: underfitting (large $\lambda$).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, model in zip(axes, ["ridge", "reg logistic"]):
    rs = [r for r in results if r["model"] == model]
    lams = [float(r["param"].split("=")[1]) for r in rs]
    ax.semilogx(lams, [r["train_f1"] for r in rs], "o-", label="train")
    ax.errorbar(lams, [r["f1"] for r in rs], yerr=[r["f1_std"] for r in rs], fmt="o-", label="validation")
    ax.set_xlabel("lambda")
    ax.set_title(model)
    ax.legend()
axes[0].set_ylabel("F1")
fig.tight_layout()
fig.savefig("results/f1_vs_lambda.png", dpi=150)
plt.show()

## Figure: F1 vs decision threshold (best model)

In [ ]:
best = max(results, key=lambda r: r["f1"])
thresholds, f1s = best["curve"]
share_positive = 1 - np.linspace(0.50, 0.99, 99)  # fraction of people predicted positive per threshold

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(share_positive * 100, f1s)
ax.axvline(y01.mean() * 100, color="gray", linestyle="--", label=f"share of CVD in the data ({y01.mean():.1%})")
ax.axvline(share_positive[np.argmax(f1s)] * 100, color="tab:red", linestyle=":", label="best threshold")
ax.set_xlabel("% of people predicted positive (lower threshold →)")
ax.set_ylabel("F1 (out-of-fold)")
ax.set_title(f"Threshold tuning: {best['model']} {best['param']}")
ax.legend()
fig.tight_layout()
fig.savefig("results/f1_vs_threshold.png", dpi=150)
plt.show()

## feature-importance

In [ ]:
# train the best regularized logistic model on all data and show its largest weights
w_full, _ = reg_logistic_regression(y01, tx, best_lam, w0, MAX_ITERS, step_size(tx, "logistic", best_lam))
top = np.argsort(-np.abs(w_full[1:]))[:15] + 1  # skip the bias (index 0)

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(np.array(features)[top][::-1], w_full[top][::-1],
        color=np.where(w_full[top][::-1] > 0, "tab:red", "tab:blue"))
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("weight (red = raises CVD risk)")
ax.set_title("Largest logistic regression weights")
fig.tight_layout()
fig.savefig("results/top_weights.png", dpi=150)
plt.show()

## Caught vs missed CVD cases

In [ ]:
# Caught vs missed: among the people who really have CVD, how do the ones the model catches
# differ from the ones it misses? Best logistic model, trained on folds 2-5, analysed on fold 1.
r = max(
    (r for r in results if "logistic" in r["model"] and "undersampled" not in r["model"]),
    key=lambda r: r["f1"],
)
s = dict(p.split("=") for p in r["param"].split() if "=" in p)
lam, pw, degree = float(s.get("lambda", 0)), float(s.get("pw", 1)), int(s.get("degree", 1))

x_all = add_powers(tx, features, degree)[0] if degree > 1 else tx
g = step_size(x_all, "logistic", lam, y01, pw)


def train(yp, y01_, x):
    return weighted_reg_logistic_regression(y01_, x, lam, np.zeros(x.shape[1]), MAX_ITERS, g, pw)


w, te, _ = cross_validation(y, y01, x_all, k_indices, 0, train)
pred = predict(x_all[te] @ w, r["threshold"])

is_cvd = y[te] == 1
caught = te[is_cvd & (pred == 1)]  # CVD cases the model predicts as CVD
missed = te[is_cvd & (pred == -1)]  # CVD cases the model misses
print(f"{r['model']} {r['param']}")
print(f"caught {len(caught)} of {len(caught) + len(missed)} CVD cases (recall {len(caught) / (len(caught) + len(missed)):.1%})")

# difference of feature means (caught - missed), divided by each feature's std so all features are comparable
std = tx.std(axis=0)
std[std == 0] = 1
diff = (tx[caught].mean(axis=0) - tx[missed].mean(axis=0)) / std
diff[0] = 0  # ignore the bias column
top = np.argsort(-np.abs(diff))[:15][::-1]

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(np.array(features)[top], diff[top], color=np.where(diff[top] > 0, "tab:red", "tab:blue"))
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("mean difference, caught − missed (in std units)")
ax.set_title("CVD cases: caught vs missed by the model")
fig.tight_layout()
fig.savefig("results/caught_vs_missed.png", dpi=150)
plt.show()

## Settings for `run.py`

Only copy these from a run with `QUICK = False`.

In [ ]:
print(f"best: {best['model']} {best['param']}  F1 {best['f1']:.4f} ± {best['f1_std']:.4f}")
if QUICK:
    print("(quick mode: set QUICK = False and re-run before copying settings into run.py)")
settings = dict(p.split("=") for p in best["param"].split() if "=" in p)
extended = ("poly reg logistic", "interact reg logistic")
model = "weighted reg logistic" if best["model"] in extended else best["model"]
print("\nsettings to copy into run.py:")
print(f'MODEL = "{model}"')
print(f"LAMBDA = {settings.get('lambda', '0')}")
print(f"POS_WEIGHT = {settings.get('pw', '1')}")
print(f"DEGREE = {settings.get('degree', '1')}")
print(f"INTERACTIONS = {settings.get('interactions', '0') == '1'}")
print(f"THRESHOLD = {best['threshold']:.6f}")